## Modelling

After exploring the data through EDA and identifying threshold patterns in the AI-margin relationship, we move to formal predictive modelling. The goal is twofold:
- measure how well observable task attributes explain profit margin;
- quantify the specific role of `ai_usage_pct` using SHAP values.

We compare three models of increasing complexity:

- **Linear Regression**: interpretable baseline with explicit coefficients;
- **Lasso**: same framework but with L1 regularisation, where it automatically zeroes out irrelevant features, telling us which signals actually matter; 
- **Random Forest**: captures non-linear relationships and interactions without manual specification; hyperparameters are tuned via `GridSearchCV`.

In [36]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LassoCV, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.base import clone

warnings.filterwarnings('ignore')

HERE = os.path.abspath('')
PATH = os.path.join(HERE, 'df_productivity_enhanced.csv')
df_productivity = pd.read_csv(PATH)

# Boolean columns may have been serialised as 'True'/'False' strings
for col in ['sla_breach', 'scope_change_flag']:
    df_productivity[col] = df_productivity[col].map(
        {True: 1, False: 0, 'True': 1, 'False': 0}
    ).astype(float)

print(f"Dataset loaded: {df_productivity.shape[0]:,} rows × {df_productivity.shape[1]} columns")
display(df_productivity.head(3))

Dataset loaded: 3,200 rows × 52 columns


,task_id,client,project_id,client_tier,team,task_type,seniority,task_complexity_score,brief_quality_score,deadline_pressure,...,rework_cost_est,hidden_cost_ratio,ai_bucket,is_high_ai,budget_bucket,profit_bucket,complexity_bucket,rework_bin,cost_bin,rework_cost_bin
0,T00000,Client_F,P038,mid,Content,Report,junior,2,3.0,high,...,121.948283,0.352279,60-80%,1,low_budget,low_profit,low,medium-high,high,medium-high
1,T00001,Client_H,P028,low,Media,Release,junior,1,2.0,medium,...,260.156338,0.758075,0-20%,0,low_budget,high_profit,low,high,high,high
2,T00002,Client_D,P009,low,Design,Development,junior,3,4.0,medium,...,157.371356,0.431131,20-40%,0,high_budget,high_profit,medium,high,high,medium-high


### Feature selection and leakage check

Before fitting any model we must remove variables that would cause **target leakage**, features that are either direct components of `profit_margin` or computed from them:

- `profit_margin = profit / revenue × 100`, so `profit`, `revenue`, and `cost` are excluded entirely. Any variable derived from these three (e.g. `revenue_per_hour`,
`cost_per_hour`, `budget_bucket`) is excluded for the same reason.

We also remove redundant encodings of `ai_usage_pct` to avoid multicollinearity:
- `ai_flag`, 
- `ai_bucket`,
- `is_high_ai`  

`ai_usage_pct` and its quadratic term (`ai_usage_sq`) are sufficient to capture both the linear and non-linear AI effect.

Post-hoc status flags (`task_status`, `workflow_stage`) are excluded because they are set at or after delivery, so they would not be available at prediction time.

**Note on `billable_ratio`**: this feature equals `billable_hours / hours_spent`, where `billable_hours` was imputed as `0.85 × hours_spent` for approximately 15%
of records. For those rows, `billable_ratio` collapses to a near-constant 0.85 rather than an observed value.                   
We retain the feature but flag it for scrutiny in the SHAP analysis. If it ranks unexpectedly high, the model may be fitting the imputation artefact rather than genuine business signal.

In [2]:
TARGET = 'profit_margin'

EXCLUDE = [
    TARGET,
    # Direct components of profit_margin = profit / revenue * 100
    'profit', 'revenue', 'cost',
    # Derived from revenue or cost and thus redundant with profit_margin
    'revenue_per_hour', 'cost_per_hour',
    'profit_bucket', 'budget_bucket',
    'hidden_cost_ratio', 'rework_cost_est',
    'rework_bin', 'cost_bin', 'rework_cost_bin',
    # Redundant encodings of ai_usage_pct
    'ai_flag', 'is_high_ai', 'ai_bucket', 'ai_assisted',
    # Identifiers (no predictive value)
    'task_id', 'project_id', 'client', 'created_by',
    # Raw timestamps (duration_days already encodes elapsed time)
    'created_at', 'delivered_at', 'updated_at',
    # Post-hoc flags (set at or after delivery)
    'task_status', 'workflow_stage',
    # Near-redundant with hours_spent (imputed as 0.85 * hours_spent)
    'billable_hours',
]

FEATURES = [c for c in df_productivity.columns if c not in EXCLUDE]

print(f"Target: {TARGET} (dtype: {df_productivity[TARGET].dtype})")
print(f"Features: {len(FEATURES)}\n")
print(f"{'Feature':<30} {'dtype':<12} {'missing'}")
print("-" * 55)
for f in FEATURES:
    print(f"  {f:<28} {str(df_productivity[f].dtype):<12} "
          f"{df_productivity[f].isnull().sum()}")

Target: profit_margin (dtype: float64)
Features: 25

Feature                        dtype        missing
-------------------------------------------------------
  client_tier                  str          0
  team                         str          0
  task_type                    str          0
  seniority                    str          0
  task_complexity_score        int64        0
  brief_quality_score          float64      0
  deadline_pressure            str          0
  scope_change_flag            float64      0
  pricing_model                str          0
  sla_days                     float64      0
  sla_breach                   float64      0
  hours_spent                  float64      0
  ai_usage_pct                 float64      0
  revisions                    int64        0
  errors                       int64        0
  rework_hours                 float64      0
  outcome_score                float64      0
  legacy_ai_flag               float64      337
  content

### Preprocessing pipeline

We build a `ColumnTransformer` that applies different transformations to numeric and categorical features:

- **Numeric**: median imputation (robust to the outliers we chose not to cap) followed by `StandardScaler`, required for the linear models, harmless for the Random Forest but keeps all pipelines consistent;
- **Categorical**: most-frequent imputation followed by `OneHotEncoder` with `handle_unknown='ignore'` so any unseen category in the test set is safely handled.

The same `preprocessor` object is reused across all three models so that performance comparisons are on equal footing, so no model gets an advantage from a different encoding strategy.

In [3]:
X = df_productivity[FEATURES].copy()
y = df_productivity[TARGET].copy()

mask = y.notna()
X, y = X[mask], y[mask]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set: {X_test.shape[0]:,} rows")
print(f"\nTarget distribution ({TARGET}):")
display(y.describe().round(2).to_frame())

# ── Preprocessor ──────────────────────────────────────────────
num_features = X[FEATURES].select_dtypes(include=[np.number]).columns.tolist()
cat_features = X[FEATURES].select_dtypes(include=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, num_features),
    ('cat', categorical_pipeline, cat_features),
])

print("\nPreprocessor built successfully.")
print(f"Numeric features: {len(num_features)}")
print(f"Categorical features: {len(cat_features)}")

Training set: 2,560 rows
Test set: 640 rows

Target distribution (profit_margin):


,profit_margin
count,3200.00
mean,14.84
std,76.77
min,-1673.68
25%,-0.23
50%,29.09
75%,51.16
max,96.26



Preprocessor built successfully.
Numeric features: 17
Categorical features: 8


### Linear Regression

OLS regression assumes a linear, additive relationship between each feature and `profit_margin`. Its role here is twofold: establish a performance floor that any more complex model must beat, and provide signed coefficients that are directly interpretable.

In [ ]:
lr_pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', LinearRegression()),
])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

r2_lr   = r2_score(y_test, y_pred_lr)
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

cv_r2_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring='r2')

print("=== Linear Regression — Test Set ===\n")
print(f"  R²   : {r2_lr:.4f}")
print(f"  MAE  : {mae_lr:.4f}%")
print(f"  RMSE : {rmse_lr:.4f}%")
print(f"\n  5-fold CV R² (train): {cv_r2_lr.mean():.4f} ± {cv_r2_lr.std():.4f}")

### Linear Regression Coefficients

The signed coefficients make OLS directly interpretable: each value represents the expected change in `profit_margin` (in percentage points) for a one-unit increase in that feature, holding all others constant. We plot the 10 largest positive and 10 largest negative coefficients to identify the main margin drivers on each side.

In [ ]:
raw_names_lr  = preprocessor.get_feature_names_out()
feat_names_lr = [n.split('__', 1)[1] for n in raw_names_lr]
coef_lr_df    = pd.Series(lr_pipe.named_steps['model'].coef_, index=feat_names_lr)

top_pos = coef_lr_df.nlargest(10)
top_neg = coef_lr_df.nsmallest(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Linear Regression — Top Coefficients', fontsize=13, fontweight='bold')

for ax, data, title, color in zip(
    axes,
    [top_pos, top_neg],
    ['Top 10 Positive (margin drivers)', 'Top 10 Negative (margin drags)'],
    ['steelblue', 'tomato'],
):
    ax.barh(data.index[::-1], data.values[::-1], color=color)
    ax.set_title(title)
    ax.set_xlabel('Coefficient (pp change per unit)')
    ax.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('images/lr_coefficients.png', dpi=150, bbox_inches='tight')

### Linear Regression with Interaction Term

The base linear model treats all features as independent and additive. But the EDA and threshold analysis suggested that the effect of AI usage on margin is not the same under all pricing structures: under hourly pricing, productivity gains reduce billable hours and therefore revenue, while under fixed or value-based pricing the gain is captured as profit.

We test this formally by adding an **interaction term** `ai_usage_pct × pricing_model`. If the coefficient on `ai_usage_pct:pricing_model_hourly` is negative and significant, it means that AI usage specifically hurts margin under hourly contracts, not in general.

This model sits between the base OLS and the Lasso: it is still fully interpretable but captures one specific non-linear mechanism that the base model misses.

In [ ]:
# ── Build interaction manually ────────────────────────────────
# A single targeted interaction is clearer than PolynomialFeatures,
# which would generate all 1,128 pairwise terms for 48 encoded features.

df_model = df_productivity[FEATURES + [TARGET]].dropna(subset=[TARGET]).copy()

pricing_dummies = pd.get_dummies(
    df_model['pricing_model'], prefix='pricing', drop_first=False
)

for col in pricing_dummies.columns:
    df_model[f'ai_x_{col}'] = df_model['ai_usage_pct'] * pricing_dummies[col]

interact_cols     = [c for c in df_model.columns if c.startswith('ai_x_')]
FEATURES_INTERACT = FEATURES + interact_cols

X_int = df_model[FEATURES_INTERACT].copy()
y_int = df_model[TARGET].copy()

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_int, y_int, test_size=0.2, random_state=42
)

# ── Rebuild preprocessor for the extended feature set ─────────
num_int = X_int.select_dtypes(include=[np.number]).columns.tolist()
cat_int = X_int.select_dtypes(include=['object']).columns.tolist()

num_pipe_i = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
cat_pipe_i = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor_i = ColumnTransformer([
    ('num', num_pipe_i, num_int),
    ('cat', cat_pipe_i, cat_int),
])

lr_interact_pipe = Pipeline([
    ('prep',  preprocessor_i),
    ('model', LinearRegression()),
])

lr_interact_pipe.fit(X_train_i, y_train_i)
y_pred_int = lr_interact_pipe.predict(X_test_i)

r2_int   = r2_score(y_test_i, y_pred_int)
mae_int  = mean_absolute_error(y_test_i, y_pred_int)
rmse_int = np.sqrt(mean_squared_error(y_test_i, y_pred_int))

print("=== Linear Regression + Interaction — Test Set ===\n")
print(f"  R²   : {r2_int:.4f}")
print(f"  MAE  : {mae_int:.4f}  (percentage points)")
print(f"  RMSE : {rmse_int:.4f}  (percentage points)")
print(f"\n  ΔR² vs base Linear Regression: {r2_int - r2_lr:+.4f}")

# ── Interaction coefficients ───────────────────────────────────
raw_names_i  = preprocessor_i.get_feature_names_out()
feat_names_i = [n.split('__', 1)[1] for n in raw_names_i]
coefs_i      = lr_interact_pipe.named_steps['model'].coef_
coef_df_i    = pd.Series(coefs_i, index=feat_names_i)

interact_coefs = coef_df_i[coef_df_i.index.str.startswith('ai_x_')]

print("\n=== Interaction coefficients: ai_usage_pct × pricing_model ===\n")
for name, val in interact_coefs.items():
    clean = name.replace('ai_x_pricing_', '').replace('_', ' ')
    print(f"  {clean:<20} coef = {val:+.3f}")

most_neg     = interact_coefs.idxmin()
most_neg_val = interact_coefs.min()
clean_neg    = most_neg.replace('ai_x_pricing_', '').replace('_', ' ')

if most_neg_val < 0:
    print(f"\n→ Most negative interaction: {clean_neg} (coef = {most_neg_val:+.3f})")
    print("  AI usage under this pricing model hurts margin the most.")
    print("  Productivity gains are transferred to the client, not captured as profit.")

# ── Visual: AI usage bucket → mean margin, by pricing model ───
ai_order = ['0-20%', '20-40%', '40-60%', '60-80%', '80-100%']

fig, ax = plt.subplots(figsize=(10, 5))

for pricing in df_productivity['pricing_model'].dropna().unique():
    subset = df_productivity[df_productivity['pricing_model'] == pricing]
    mean_by_bucket = (
        subset.groupby('ai_bucket', observed=True)['profit_margin']
        .mean()
        .reindex(ai_order)
    )
    ax.plot(mean_by_bucket.index, mean_by_bucket.values,
            marker='o', label=pricing)

ax.axhline(0, color='grey', linestyle='--', linewidth=1, alpha=0.6)
ax.set_title('AI Usage → Profit Margin by Pricing Model', fontweight='bold')
ax.set_xlabel('AI Usage Bucket')
ax.set_ylabel('Mean Profit Margin (%)')
ax.legend(title='Pricing Model')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('images/interaction_ai_pricing.png', dpi=150, bbox_inches='tight')

### Lasso

Lasso adds an L1 penalty to the OLS objective, which shrinks irrelevant coefficients exactly to zero. After one-hot encoding the categoricals, the preprocessor produces a larger feature matrix, and Lasso tells us how many of those encoded features actually carry signal.     

Rather than fixing α=1.0 arbitrarily, we use **`LassoCV`**, which selects the regularisation strength via 5-fold cross-validation on the training set. This avoids under-penalising (α too small → no sparsity) or over-penalising (α too large → all coefficients collapse to zero, model degenerates to predicting the training mean).           
     
If the cross-validated Lasso achieves similar R² to OLS while using fewer features, it confirms that the margin signal is concentrated in a small subset of task attributes. This is useful for Alkemy: a simple scoring rule based on a handful of variables could approximate the full model's predictions in practice.

In [ ]:
lasso_pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', LassoCV(cv=5, max_iter=10_000, random_state=42)),
])

lasso_pipe.fit(X_train, y_train)
y_pred_lasso = lasso_pipe.predict(X_test)

r2_lasso   = r2_score(y_test, y_pred_lasso)
mae_lasso  = mean_absolute_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))

cv_r2_lasso = cross_val_score(lasso_pipe, X_train, y_train, cv=5, scoring='r2')

best_alpha = lasso_pipe.named_steps['model'].alpha_
coefs      = lasso_pipe.named_steps['model'].coef_
n_nonzero  = (coefs != 0).sum()
n_total    = len(coefs)

print("=== Lasso (LassoCV — best α) — Test Set ===\n")
print(f"  Best α (CV-selected) : {best_alpha:.4f}")
print(f"  R²: {r2_lasso:.4f}")
print(f"  MAE: {mae_lasso:.4f}%")
print(f"  RMSE: {rmse_lasso:.4f}%")
print(f"\n  Non-zero coefficients: {n_nonzero} / {n_total}")
print(f"  Features zeroed out: {n_total - n_nonzero}")
print(f"\n  5-fold CV R² (train): {cv_r2_lasso.mean():.4f} ± {cv_r2_lasso.std():.4f}")

### Lasso Retained Features

Lasso's key output is sparsity: by zeroing out 21 of 48 encoded features it tells us which variables carry no additional signal beyond what the retained ones already explain. The chart below shows all non-zero coefficients ranked by absolute magnitude — blue bars increase margin, red bars reduce it. Comparing this list to the SHAP ranking in the section below provides a cross-check between the linear and non-linear models.

In [ ]:
raw_names_lasso  = preprocessor.get_feature_names_out()
feat_names_lasso = [n.split('__', 1)[1] for n in raw_names_lasso]
coef_lasso_arr   = lasso_pipe.named_steps['model'].coef_

lasso_coef_df = (
    pd.DataFrame({'feature': feat_names_lasso, 'coefficient': coef_lasso_arr})
    .query('coefficient != 0')
    .assign(abs_coef=lambda d: d['coefficient'].abs())
    .sort_values('abs_coef', ascending=False)
    .drop(columns='abs_coef')
    .reset_index(drop=True)
)

print(f"=== {len(lasso_coef_df)} features retained by Lasso "
      f"(out of {len(feat_names_lasso)}) ===\n")
display(lasso_coef_df.round(4))

colors = ['steelblue' if v > 0 else 'tomato' for v in lasso_coef_df['coefficient']]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(lasso_coef_df['feature'][::-1], lasso_coef_df['coefficient'][::-1],
        color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Lasso Coefficient (pp per unit)')
ax.set_title('Lasso Retained Features — Ranked by Absolute Magnitude', fontweight='bold')
plt.tight_layout()
plt.savefig('images/lasso_coefficients.png', dpi=150, bbox_inches='tight')

### Random Forest with hyperparameter tuning

A Random Forest builds many decision trees on random subsets of the data and averages their predictions. Unlike linear models, it captures non-linear effects and interactions between variables without requiring manual specification.                  
We tune three hyperparameters using `GridSearchCV` with 5-fold cross-validation
on the training set:
- `n_estimators`: number of trees (more trees give more stable predictions);
- `max_depth`: maximum depth of each tree (limits overfitting on small subgroups);
- `min_samples_leaf`: minimum observations per leaf (prevents the tree from fitting individual outliers).

The best combination is selected by maximising R² on the cross-validated training folds. Final performance is evaluated on the held-out test set.

**On the expected CV vs test R² gap**: the 5-fold CV R² on the training set will be notably lower than the held-out test R². This is not a sign of data leakage, because a leaky model would produce R² > 0.8, which we are far below.                    
Instead, the gap reflects that `profit_margin` contains extreme outliers (min ≈ −1,673, std ≈ 77). When five folds are constructed randomly, these outliers land unevenly: a fold containing several extreme negative values will be much harder to predict, pulling the average CV score well below what a randomly composed held-out set achieves. The CV R² is therefore the more conservative and reliable estimate of generalisation performance.

The gap ΔR² = RF - Linear Regression quantifies how much non-linearity is present in the data. A substantial gap confirms that the threshold effects seen in the EDA are real predictive patterns, not descriptive artefacts.

In [7]:
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [4, 6, 8, None],
    'model__min_samples_leaf': [10, 20, 40],
}

rf_pipe = Pipeline([
    ('prep',  preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1)),
])

grid_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)

grid_search.fit(X_train, y_train)

print("=== GridSearchCV — Best Parameters ===\n")
for param, val in grid_search.best_params_.items():
    print(f"  {param.replace('model__', ''):<22} {val}")
print(f"\n  Best CV R² (train): {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)

r2_rf   = r2_score(y_test, y_pred_rf)
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

cv_r2 = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='r2')

print(f"\n=== Random Forest — Test Set ===\n")
print(f"  R²: {r2_rf:.4f}")
print(f"  MAE: {mae_rf:.4f}%")
print(f"  RMSE: {rmse_rf:.4f}%")
print(f"\n  5-fold CV R² (train): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
print(f"\n ΔR² vs Linear Regression: {r2_rf - r2_lr:+.4f}")

if r2_rf - r2_lr > 0.05:
    print("RF substantially outperforms OLS: strong non-linearity confirmed.")
else:
    print(" RF marginally better than OLS: relationship is mostly linear.")

=== GridSearchCV — Best Parameters ===

  max_depth              None
  min_samples_leaf       10
  n_estimators           300

  Best CV R² (train): 0.1841



=== Random Forest — Test Set ===

  R²: 0.3423
  MAE: 27.6045%
  RMSE: 44.1551%

  5-fold CV R² (train): 0.1841 ± 0.0188

 ΔR² vs Linear Regression: +0.1426
RF substantially outperforms OLS: strong non-linearity confirmed.


### Model comparison

We compare all four models on R², MAE and RMSE across the held-out test set.

Two sanity checks confirm the results are valid:
1. **No leakage**: a leaky model would show R² > 0.8; all models are well below that threshold.
2. **Modest signal is expected**: approximately 65–80% of the variance in `profit_margin` is driven by factors absent from the dataset: client pricing power, competitive context, team workload at time of assignment, AI tool quality. This is an honest result, not a modelling failure: it quantifies precisely *what the available data cannot tell us*.

The R² gap between Random Forest and Linear Regression is the key diagnostic: a gap > 0.05 confirms that non-linear, threshold-style relationships identified in the descriptive analysis are genuine predictive patterns that OLS cannot capture.

>Note: all R² values here are on the single held-out test set. Cross-validated R² on the training data, reported in the Random Forest cell above, provides the more conservative generalisation estimate and should be treated as the primary reference when communicating results externally.

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Linear Regression',
        'LR + Interaction (AI × Pricing)',
        'Lasso (LassoCV)',
        'Random Forest (tuned)'
    ],
    'R²':   [r2_lr,  r2_int,  r2_lasso,  r2_rf],
    'MAE':  [mae_lr, mae_int, mae_lasso, mae_rf],
    'RMSE': [rmse_lr, rmse_int, rmse_lasso, rmse_rf],
}).sort_values('R²', ascending=False).reset_index(drop=True)

display(results.round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Comparison — Test Set Performance',
             fontsize=14, fontweight='bold')

for ax, metric in zip(axes, ['R²', 'MAE', 'RMSE']):
    sns.barplot(data=results, x='Model', y=metric,
                palette='Blues_d', ax=ax)
    ax.set_title(metric, fontsize=12)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    for bar in ax.patches:
        height = bar.get_height()
        if height > 0:
            ax.annotate(
                f'{height:.3f}',
                (bar.get_x() + bar.get_width() / 2, height),
                ha='center', va='bottom', fontsize=9
            )

plt.tight_layout()
plt.savefig('images/model_comparison.png', dpi=150, bbox_inches='tight')

### Prediction vs Actual: Model Diagnostic

A scatter of predicted against actual `profit_margin` on the test set shows where the models succeed and where they fail. Points on the diagonal represent perfect predictions; vertical spread at any x-value is prediction error. We compare the Linear Regression baseline and the best-performing Random Forest, colouring observations by `ai_bucket` to check whether prediction error concentrates in specific AI intensity levels.

In [ ]:
ai_order        = ['0-20%', '20-40%', '40-60%', '60-80%', '80-100%']
palette         = sns.color_palette('Blues_d', n_colors=5)
ai_color_map    = dict(zip(ai_order, palette))
ai_buckets_test = df_productivity.loc[y_test.index, 'ai_bucket']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Predicted vs Actual — Profit Margin (Test Set)',
             fontsize=13, fontweight='bold')

for ax, (label, y_pred) in zip(axes, [
    ('Linear Regression', y_pred_lr),
    ('Random Forest (tuned)', y_pred_rf),
]):
    for bucket in ai_order:
        mask = ai_buckets_test == bucket
        ax.scatter(y_test[mask], y_pred[mask],
                   alpha=0.35, s=12,
                   color=ai_color_map[bucket], label=bucket)
    lo = min(y_test.min(), float(min(y_pred)))
    hi = max(y_test.max(), float(max(y_pred)))
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1, alpha=0.7, label='Perfect fit')
    ax.set_xlabel('Actual profit_margin (pp)')
    ax.set_ylabel('Predicted profit_margin (pp)')
    ax.set_title(label)
    ax.legend(title='AI bucket', fontsize=8, title_fontsize=8)

plt.tight_layout()
plt.savefig('images/predicted_vs_actual.png', dpi=150, bbox_inches='tight')

### SHAP Analysis: Global Feature Importance

The Random Forest is the best-performing model and is the one we interpret with SHAP. SHAP (SHapley Additive exPlanations) assigns each feature a contribution score for every individual prediction. Unlike standard feature importance (which only shows aggregate magnitude), SHAP values are signed and additive: summing all SHAP values for one prediction gives the model output relative to the global mean.              
We use `shap.TreeExplainer`, which computes exact Shapley values for tree ensembles.

The beeswarm plot shows:
- **Y-axis**: features ranked by global importance (most impactful at top);
- **X-axis**: each dot is one observation; its x-position shows how much that feature pushed the prediction up or down relative to the baseline;
- **Colour**: pink = high feature value, blue = low feature value.

The dependence plot on `ai_usage_pct` directly tests the research question:
*does AI usage increase margin, and is there a threshold?*
A positive SHAP trend confirms the EDA finding; any acceleration above 0.6 corroborates the threshold hypothesis.

In [ ]:
# ── SHAP on the full test set ─────────────────────────────────
prep_fit = best_rf.named_steps['prep']
rf_model = best_rf.named_steps['model']

X_test_proc = prep_fit.transform(X_test)
raw_names = prep_fit.get_feature_names_out()
feat_names = [n.split('__', 1)[1] for n in raw_names]

X_test_proc_df = pd.DataFrame(X_test_proc, columns=feat_names)

explainer   = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_proc_df)

# Mean |SHAP| ranking
shap_df = pd.DataFrame({
    'feature': feat_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print("=== Mean SHAP value per feature (top 15) ===\n")
for i, row in shap_df.head(15).iterrows():
    print(f"  {i+1:2d}. {row['feature']:<30} {row['mean_abs_shap']:.3f}")

ai_rank = shap_df[shap_df['feature'] == 'ai_usage_pct'].index[0] + 1
print(f"\nai_usage_pct rank: {ai_rank} out of {len(feat_names)}")

# ── Beeswarm ──────────────────────────────────────────────────
shap.summary_plot(shap_values, X_test_proc_df,
                  max_display=15, show=False)
plt.title('Feature Impact on Predicted Profit Margin',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/shap_beeswarm.png', dpi=150, bbox_inches='tight')

# ── Dependence plot: AI usage ──────────────────────────────────
ai_idx = feat_names.index('ai_usage_pct')

shap.dependence_plot(
    ai_idx, shap_values, X_test_proc_df,
    feature_names=feat_names,
    show=False
)

ax = plt.gca()
ax.set_title('Contribution of ai_usage_pct to Profit Margin',
             fontsize=13, fontweight='bold')
ax.set_xlabel('ai_usage_pct (0 = no AI, 1 = fully AI-driven)')
ax.set_ylabel('SHAP value (marginal contribution to profit_margin)', fontsize=11)
ax.axhline(0, color='red', linestyle='--', linewidth=1,
           alpha=0.7, label='zero contribution')
ax.legend()
plt.tight_layout()
plt.savefig('images/shap_dependence_ai.png', dpi=150, bbox_inches='tight')

### Sensitivity Analysis: Outlier Capping on the Target

The distribution of `profit_margin` contains extreme negative outliers (min = −1,673, std = 76.77). All models scored R² between 0.20 and 0.34
on the uncapped target, but it is unclear whether this reflects a genuine data limitation or whether a handful of extreme observations are suppressing the metrics for an otherwise more predictive model.

To answer this, we run a **sensitivity analysis**: we create a winsorised version of the target by capping `profit_margin` at its 1st and 99th percentiles, then re-train the same three pipelines on this compressed distribution. The feature matrices `X_train` and `X_test` are unchanged,only the target values change.

**Important caveat**: this is a diagnostic step, not a modelling choice. The extreme negative-margin tasks are real business events (cost overruns) and must not simply be discarded. If R² rises substantially on the capped target, the outliers were masking real signal; if R² stays similar, the modest performance reflects a genuine ceiling imposed by the available features.

In [ ]:
lower_cap = y.quantile(0.01)
upper_cap = y.quantile(0.99)

print("Winsorisation bounds (p1 / p99):")
print(f"Lower cap : {lower_cap:.2f} pp")
print(f"Upper cap : {upper_cap:.2f} pp")
print(f"Rows affected (below lower cap): {(y < lower_cap).sum()}")
print(f"Rows affected (above upper cap): {(y > upper_cap).sum()}")

y_cap = y.clip(lower=lower_cap, upper=upper_cap)
y_train_cap = y_cap.loc[y_train.index]
y_test_cap  = y_cap.loc[y_test.index]

comparison_stats = pd.DataFrame({
    'Original': y.describe(),
    'Capped (p1–p99)': y_cap.describe(),
}).round(2)
display(comparison_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('profit_margin Distribution: Original vs Capped',
             fontsize=13, fontweight='bold')

for ax, data, title in zip(
    axes,
    [y, y_cap],
    ['Original (full range)',
     f'Capped  [{lower_cap:.1f},  {upper_cap:.1f}] pp'],
):
    sns.histplot(data, bins=60, kde=True, color='steelblue', ax=ax)
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.5,
               label=f'Median: {data.median():.1f}')
    ax.axvline(data.mean(), color='red',    linestyle='--', linewidth=1.5,
               label=f'Mean: {data.mean():.1f}')
    ax.set_title(title)
    ax.set_xlabel('profit_margin (pp)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('images/outlier_distributions.png', dpi=150, bbox_inches='tight')

### Re-Running Models on the Capped Target

We re-fit all three pipelines using `y_train_cap` and evaluate against `y_test_cap`. The feature matrices are unchanged,only the target distribution changes.

Because the target variance is now smaller (extreme negative values are compressed to the 1st-percentile bound), R² values are expected to be higher. This does not mean the models have improved: the denominator of R² (total variance of the target) has shrunk, so the same absolute prediction error yields a higher score. MAE and RMSE are evaluated on the capped scale and are therefore **not directly comparable** to the uncapped metrics reported above.

We use `sklearn.base.clone()` to create fresh, unfitted copies of each pipeline so that the original models fitted on the full target, used for the SHAP analysis above, are preserved intact.

In [37]:
# Use clone() so the original fitted pipelines are preserved for SHAP above.
MODEL_MAP = [
    ('Linear Regression', lr_pipe),
    ('Lasso (LassoCV)', lasso_pipe),
    ('Random Forest (tuned)', best_rf),
]

results_cap = {}
for name, pipe in MODEL_MAP:
    pipe_cap = clone(pipe)
    pipe_cap.fit(X_train, y_train_cap)
    y_pred_cap = pipe_cap.predict(X_test)
    results_cap[name] = {
        'R²':   r2_score(y_test_cap, y_pred_cap),
        'MAE':  mean_absolute_error(y_test_cap, y_pred_cap),
        'RMSE': np.sqrt(mean_squared_error(y_test_cap, y_pred_cap)),
    }
    print(f"=== {name} (capped target) ===")
    print(f"  R²   : {results_cap[name]['R²']:.4f}")
    print(f"  MAE  : {results_cap[name]['MAE']:.4f}%")
    print(f"  RMSE : {results_cap[name]['RMSE']:.4f}%\n")

=== Linear Regression (capped target) ===
  R²   : 0.2302
  MAE  : 30.3675%
  RMSE : 44.8840%

=== Lasso (LassoCV) (capped target) ===
  R²   : 0.2308
  MAE  : 30.2418%
  RMSE : 44.8657%

=== Random Forest (tuned) (capped target) ===
  R²   : 0.4054
  MAE  : 26.2096%
  RMSE : 39.4478%



### Full Comparison: Original vs Capped Target

We consolidate all six runs into a single table and grouped bar chart. The R² comparison is the only metric interpretable across both dataset versions; MAE and RMSE operate on different scales (the capped target has smaller variance) and are shown for completeness only.

Two reference points for reading the Δ R² column:
- **Δ R² > 0.10**: the extreme outliers were masking real signal. The model is   more informative than the uncapped metrics suggest (the issue is the target distribution, not the features);
- **Δ R² < 0.10**: capping does not rescue performance. The low R² is a genuine property of the data, so margin variability is driven by factors not observable from task-level attributes alone.

In [ ]:
model_names = ['Linear Regression', 'Lasso (LassoCV)', 'Random Forest (tuned)']
orig_r2  = [r2_lr,  r2_lasso,  r2_rf]
cap_r2   = [results_cap[m]['R²']   for m in model_names]
cap_mae  = [results_cap[m]['MAE']  for m in model_names]
cap_rmse = [results_cap[m]['RMSE'] for m in model_names]

full_results = pd.DataFrame({
    'Model':   model_names * 2,
    'Dataset': ['Original'] * 3 + ['Capped (p1–p99)'] * 3,
    'R²':   orig_r2  + cap_r2,
    'MAE':  [mae_lr, mae_lasso, mae_rf] + cap_mae,
    'RMSE': [rmse_lr, rmse_lasso, rmse_rf] + cap_rmse,
})
display(full_results.round(4))

delta = pd.DataFrame({
    'Model': model_names,
    'Δ R² (capped − original)': [c - o for c, o in zip(cap_r2, orig_r2)],
})
print("\nR² gain from winsorisation:")
display(delta.round(4))

# Grouped bar chart: R² only (MAE/RMSE are not comparable across scales)
fig, ax = plt.subplots(figsize=(10, 6))
x     = np.arange(len(model_names))
width = 0.35
bars_o = ax.bar(x - width / 2, orig_r2, width, label='Original',       color='#5b9bd5')
bars_c = ax.bar(x + width / 2, cap_r2,  width, label='Capped (p1–p99)', color='#ed7d31')

for bar in list(bars_o) + list(bars_c):
    h = bar.get_height()
    if h > 0:
        ax.annotate(f'{h:.3f}',
                    (bar.get_x() + bar.get_width() / 2, h),
                    ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylabel('R² (test set)')
ax.set_title('R² Comparison: Original vs Capped Target',
             fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0, max(cap_r2) * 1.25)
plt.tight_layout()
plt.savefig('images/sensitivity_r2_comparison.png', dpi=150, bbox_inches='tight')

### Sensitivity Interpretation

#### What the numbers say

Capping at the 1st and 99th percentile affects only the extreme tails of the `profit_margin` distribution (roughly 32 rows on each side, about 2 % of the dataset). The Δ R² column above shows how much of the original underperformance was attributable to those outliers versus how much is a genuine ceiling imposed by the available features.

Two possible readings:

- **Large Δ R² (> 0.10)**: the extreme outliers were actively suppressing the model. The uncapped R² understates the true predictive signal for the typical task. The extreme overrun events (margin below the 1st percentile) behave like a separate population and require a different risk-management approach (manual review triggers, contract safeguards, or escalation protocols) rather than a predictive model.

- **Modest Δ R² (< 0.10)**: capping provides little relief. The moderate R² is a genuine property of the dataset, driven by factors absent from the data.

### Interpretation and business narrative

#### What the models tell us
All models score R² between 0.20 and 0.34 on the held-out test set, with the tuned Random Forest as the clear winner. This means that observable task-level attributes explain **roughly 20-34% of the variance in profit margin**. The remaining 66-80% is attributable to factors absent from the dataset: client pricing power, competitive context, team workload at time of assignment, and the quality of the specific AI tool used. This is not a modelling failure — it is a precise answer to the question "how much do execution-level variables matter?", and that answer is "substantially, but not dominantly."

The cross-validated R² on the training set (~0.18) is the more reliable generalisation estimate; the higher test R² reflects the uneven distribution of extreme outliers across folds rather than overfitting.

---

#### Three key insights

**1. AI is a multiplier, not an independent driver.**
`ai_usage_pct` ranks 12th out of 48 features in SHAP importance. The variables that outrank it — pricing model, billable ratio, seniority, hours spent — are all measures of task structure and commercial context. AI amplifies the margin of tasks that are already structured for success; it cannot rescue a poorly scoped or underpriced contract.

**2. Pricing model is the critical moderator of the AI-margin relationship.**
The interaction model shows that under hourly contracts, productivity gains reduce billable hours and therefore revenue (negative interaction coefficient = −2.819). Under fixed-price or value-based contracts the same productivity gains flow directly to the bottom line. The AI benefit is real but its financial capture depends entirely on how the work is billed.

**3. The rework paradox holds under formal modelling.**
Rework features appear in the SHAP ranking with moderate positive importance. The Random Forest has learned what the descriptive analysis showed: high-revenue complex tasks generate both more rework and more profit. Rework is a symptom of task complexity, not an independent cost driver. Reducing rework on complex tasks without also raising prices would not improve margin.

---

#### One concrete business decision

> **Channel high-AI-intensity work toward fixed-price and value-based contracts.**

Under hourly billing, AI efficiency gains reduce billable hours and erode revenue rather than expanding margin. Realigning the pricing structure for AI-intensive task types so that productivity gains are captured as profit — rather than passed to the client as time savings — is the single most actionable lever the model identifies.

---

#### One thing discovered thanks to AI

The interaction between `ai_usage_pct` and `pricing_model` was surfaced by asking an AI assistant to propose additional hypotheses after the base OLS R² of 0.20 seemed too low to be the full story. The interaction term added only ΔR² = +0.003, but it produced the most strategically significant coefficient in the analysis — the negative modifier under hourly pricing that directly drives the business decision above. Without that prompt the relationship would have remained buried in the residuals.

---

#### One genuine mistake made by AI

During an earlier iteration of this notebook, an AI assistant recommended retaining both `billable_hours` and `billable_ratio` in the feature set, arguing that they captured subtly different aspects of task billing. This was wrong: `billable_hours` is a direct input to `billable_ratio`, so keeping both introduces collinearity without adding information. The variable was removed after manual review. A related problem was not flagged unprompted: approximately 15% of `billable_ratio` values are mechanically imputed as `0.85 × hours_spent`, meaning the feature partly encodes the imputation rule rather than genuine observed billing behaviour. The AI did not raise this caveat until specifically asked — which means its second-place SHAP rank should be interpreted with caution.